In [12]:
import pandas as pd
import warnings
import sys
import os
import pickle
import numpy as np

sys.path.append(os.path.abspath('..'))

from src.app.core.api import Features
from src.app.core.model import Model

warnings.filterwarnings('ignore', category=UserWarning, module='sklearn')
pd.set_option('display.width', 200)

In [13]:
DATA_PATH = "../hw6/data/val_features.csv"

MODEL_PATH = "../hw6/models/logistic_regression.pkl"

SCALER_PATH = "../hw6/models/logistic_regression_scaler.pkl"

IDX_TO_COMPARE_COUNT = 1000

In [14]:
df_val = pd.read_csv(DATA_PATH)
print(f"Валидационная выборка: {df_val.shape}")


Валидационная выборка: (52656, 10)


In [15]:
features = Features.from_dataframe(df_val)

Логистическая регрессия была обучена на масштабированных признаках

In [16]:
with open(SCALER_PATH, 'rb') as f:
    scaler = pickle.load(f)

df_scaled = scaler.transform(df_val)

Обертка

In [17]:
wrapper = Model(MODEL_PATH, SCALER_PATH)

Модель

In [18]:
with open(MODEL_PATH, 'rb') as f:
    model = pickle.load(f)

Проверяем пробы на обертке и модели

In [19]:
probas_wrapper = wrapper.predict_proba_batch(df_val[:IDX_TO_COMPARE_COUNT])

In [20]:
probas_model = model.predict_proba(df_scaled[:IDX_TO_COMPARE_COUNT])[:, 1]

In [21]:
if np.allclose(probas_wrapper, probas_model, atol=1e-8):
    print("Все вероятности совпадают")
else:
    print("Вероятности не совпадают")

Все вероятности совпадают


Проверка калькулятора

In [24]:
idx = 18

client_data = df_val.iloc[idx:idx+1]

features = Features.from_dataframe(client_data)

result = wrapper.get_scoring_result(features)

print("Информация о клиенте")
print(client_data)

print("Результат")
print(f"Решение о выдаче займа: {result.decision}")
print(f"Сумма: {result.amount}")
print(f"threshold: {result.threshold}")
print(f"proba: {result.proba:.5f}")

Информация о клиенте
    weighted_ext_score  ext_source_3  ext_source_2  days_registration  days_birth  days_id_publish  annuity_to_income_proportion  interest_rate  days_employed  amt_annuity
18            0.437571      0.698667      0.569975            -3377.0      -21040            -4385                      0.333333      21.314007          -2228     225000.0
Результат
Решение о выдаче займа: ScoringDecision.ACCEPTED
Сумма: 302400
threshold: 0.3
proba: 0.14289
